In [ ]:
# ir_eval.py
import json, os, re
from typing import List, Dict, Any, Iterable, Tuple, Optional, Union
import numpy as np
import pandas as pd

Number = Union[int, np.integer]
NONE_VALUE = -1.0

def load_model_outputs(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        head = f.read(2048)
        f.seek(0)
        if head.lstrip().startswith("["):
            return json.load(f)
        data = []
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
        return data

_OUTPUT_TAG_RE = re.compile(r"<output>(?P<body>.*?)</output>", flags=re.IGNORECASE|re.DOTALL)
_BRACKET_ID_RE = re.compile(r"\[([A-Za-z0-9]+)\]")

def _dedup_preserve_order(ids: List[str]) -> List[str]:
    seen = set(); out = []
    for x in ids:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

def parse_rank_ids_from_text(text: str) -> List[str]:
    """
    Parse list of uppercase IDs in ranking output from model single output.
    Prefer content in <output> tag, fall back to full text.
    """
    if not text:
        return []
    body = None
    m = _OUTPUT_TAG_RE.search(text)
    if m:
        body = m.group("body")
    else:
        body = text
    ids = [s.upper() for s in _BRACKET_ID_RE.findall(body)]
    return _dedup_preserve_order(ids)

def parse_ir_predictions(model_outputs: List[Dict[str, Any]]) -> List[List[List[str]]]:
    """
    Parse the entire file into: preds[sample_idx][run_idx] = ['H','C','D',...]
    """
    all_preds: List[List[List[str]]] = []
    for sample in model_outputs:
        runs: List[List[str]] = []
        for out in sample.get("outputs", []):
            ids = parse_rank_ids_from_text((out.get("text") or "").strip())
            runs.append(ids)
        all_preds.append(runs)
    return all_preds

def parse_gt_order(gt_str: str) -> List[str]:
    """
    Example ground_truth: '[B] > [D] > [F] > ...'
    Returns: ['B','D','F', ...]
    """
    if not gt_str:
        return []
    return [s.upper() for s in _BRACKET_ID_RE.findall(gt_str)]

def get_average_tokens(model_outputs: List[Dict[str, Any]]) -> List[float]:
    """
    For each sample, take the average of num_tokens across runs; use 0 if missing.
    """
    avg_tokens: List[float] = []
    for sample in model_outputs:
        toks = [float(out.get("num_tokens", 0)) for out in sample.get("outputs", [])]
        avg_tokens.append(float(np.mean(toks)) if toks else 0.0)
    return avg_tokens

def _rank_map(order: List[str]) -> Dict[str, int]:
    """Return item->rank dict (0-based)."""
    return {x: i for i, x in enumerate(order)}

def top1_agreement(pred: List[str], gt: List[str]) -> float:
    """Return 1 if the predicted top-1 matches GT top-1 (and both are non-empty)."""
    if not pred or not gt:
        return 0.0
    return 1.0 if pred[0] == gt[0] else 0.0

def pairwise_accuracy(pred: List[str], gt: List[str]) -> float:
    """
    Pairwise order agreement: only computed on intersection of pred and gt.
    O(m^2) where m is the size of the intersection (usually <= 20).
    """
    if not pred or not gt:
        return 0.0
    rp = _rank_map(pred)
    rg = _rank_map(gt)
    common = [x for x in rp.keys() if x in rg]
    m = len(common)
    if m <= 1:
        return 0.0
    correct = 0
    total = 0
    for i in range(m):
        for j in range(i+1, m):
            a, b = common[i], common[j]
            sign_p = np.sign(rp[a] - rp[b])
            sign_g = np.sign(rg[a] - rg[b])
            if sign_p == sign_g:
                correct += 1
            total += 1
    return correct / total if total > 0 else 0.0

def spearman_footrule_similarity(pred: List[str], gt: List[str]) -> float:
    """
    Spearman footrule similarity (1 - normalized distance).
    Computed on intersection only: F = sum |r_p(x) - r_g(x)|; F_max = floor(m^2/2).
    Returns in [0,1], higher is better.
    """
    if not pred or not gt:
        return 0.0
    rp = _rank_map(pred)
    rg = _rank_map(gt)
    common = [x for x in rp.keys() if x in rg]
    m = len(common)
    if m == 0:
        return 0.0
    F = sum(abs(rp[x] - rg[x]) for x in common)
    Fmax = (m * m) // 2  # floor(m^2/2)
    if Fmax == 0:
        return 1.0
    return 1.0 - (F / Fmax)

def ndcg_from_orders(pred: List[str], gt: List[str], k: Optional[int] = None, use_exp_gain: bool = False) -> float:
    """
    Construct graded relevance from GT order only: rel(x) = M - rank_gt(x)
      - M = len(gt), rank from 0, so rel in {M, M-1, ..., 1}
      - items in pred but not in GT get rel=0
    Compute DCG/IDCG on the top k (default: full length) of pred, return NDCG in [0,1].
    """
    if not pred or not gt:
        return 0.0
    rg = _rank_map(gt)
    M = len(gt)
    rel = lambda x: (M - rg[x]) if x in rg else 0
    if k is None or k <= 0:
        k = len(pred)
    top = pred[:k]

    def _gain(r: int) -> float:
        if not use_exp_gain:
            return float(r)
        return float(2**r - 1)

    dcg = 0.0
    for i, x in enumerate(top):
        r = rel(x)
        if r > 0:
            dcg += _gain(r) / np.log2(i + 2.0)

    rels_sorted = sorted([rel(x) for x in gt], reverse=True)
    idcg = 0.0
    for i, r in enumerate(rels_sorted[:k]):
        idcg += _gain(r) / np.log2(i + 2.0)

    return float(dcg / idcg) if idcg > 0 else 0.0

def evaluate_ir(
    model_outputs: List[Dict[str, Any]],
    ground_truths: List[str],
    k_list: Iterable[int] = (5, 10)
) -> pd.DataFrame:
    """
    Return DataFrame (one row per sample):
      - Average metrics: avg_ndcg@{k} (k in k_list), avg_pairacc, avg_top1, avg_spearmanF
      - Also: per-run NDCG@10 expanded as columns ndcg10_1, ndcg10_2, ...
    """
    preds = parse_ir_predictions(model_outputs)
    max_runs = max((len(runs) for runs in preds), default=0)

    rows = []
    for i, (runs, gt_str) in enumerate(zip(preds, ground_truths)):
        gt_order = parse_gt_order(gt_str)
        ndcg_means = {k: [] for k in k_list}
        pairacc_list, top1_list, spf_list = [], [], []
        ndcg10_runs: List[float] = []

        for pred in runs:
            if not pred:
                for k in k_list: ndcg_means[k].append(0.0)
                pairacc_list.append(0.0); top1_list.append(0.0); spf_list.append(0.0)
                ndcg10_runs.append(0.0)
                continue

            for k in k_list:
                ndcg_means[k].append(ndcg_from_orders(pred, gt_order, k=k))
            pairacc_list.append(pairwise_accuracy(pred, gt_order))
            top1_list.append(top1_agreement(pred, gt_order))
            spf_list.append(spearman_footrule_similarity(pred, gt_order))
            ndcg10_runs.append(ndcg_from_orders(pred, gt_order, k=10))

        rec = {"index": i}
        for k in k_list:
            rec[f"avg_ndcg@{k}"] = float(np.mean(ndcg_means[k])) if ndcg_means[k] else 0.0
        rec["avg_pairacc"]   = float(np.mean(pairacc_list)) if pairacc_list else 0.0
        rec["avg_top1"]      = float(np.mean(top1_list))    if top1_list    else 0.0
        rec["avg_spearmanF"] = float(np.mean(spf_list))     if spf_list     else 0.0

        if len(ndcg10_runs) < max_runs:
            ndcg10_runs = ndcg10_runs + [0.0] * (max_runs - len(ndcg10_runs))

        for j, v in enumerate(ndcg10_runs, start=1):
            rec[f"ndcg10_{j}"] = float(v)

        rows.append(rec)

    cols = (
        ["index"]
        + [f"avg_ndcg@{k}" for k in k_list]
        + ["avg_pairacc", "avg_top1", "avg_spearmanF"]
        + [f"ndcg10_{j}" for j in range(1, max_runs + 1)]
    )
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns:
            df[c] = 0.0
    return df[cols]

def summarize_dataframe(df: pd.DataFrame) -> Dict[str, float]:
    """
    Return overall metric means for quick experiment review.
    """
    out = {}
    for c in df.columns:
        if c == "index": continue
        out[c] = float(df[c].mean())
    return out

def evaluate_ir_from_files(
    output_file: str,
    loaded_test_data: List[Dict[str, Any]],
    k_list: Iterable[int] = (5, 10)
) -> Dict[str, Any]:
    """
    End-to-end: read output file -> parse predictions -> evaluate -> return results dict:
      {
        "averages_tokens": [...],
        "detail": <per-sample DataFrame as dict>,
        "summary": <overall metric averages>,
      }
    """
    model_outputs = load_model_outputs(output_file)
    gts = [ex.get("ground_truth", "") for ex in loaded_test_data]

    df = evaluate_ir(model_outputs, gts, k_list=k_list)
    avg_tokens = get_average_tokens(model_outputs)

    return {
        "averages_tokens": avg_tokens,
        "detail": df.to_dict(orient="records"),
        "summary": summarize_dataframe(df),
    }


In [ ]:
import os, json, numpy as np, pandas as pd

import pandas as pd
import numpy as np
import pickle
import os
import json

seed = 2025
dataset_name = 'rank_ir'

model = 'Qwen3-4B'
PATH = os.path.join('/data/', f'{dataset_name}/{model}')
if not os.path.exists(PATH):
    os.makedirs(PATH)
data_path = os.path.join('/data/', f'{dataset_name}/processed_data')
saved_file_path = os.path.join(PATH, 'saved_data')
saved_result_path = os.path.join(PATH, 'saved_result')
if not os.path.exists(saved_file_path):
    os.makedirs(saved_file_path)
if not os.path.exists(saved_result_path):
    os.makedirs(saved_result_path)

with open(os.path.join(data_path, "test_data.json"), "r", encoding="utf-8") as f:
    loaded_test_data = json.load(f)


def eval_ranking(system, dataset_name, mode, test_data):
    output_file = os.path.join(saved_file_path, f'{system}_{dataset_name}_{mode}_llm_output.jsonl')
    print(f'eval {system} {dataset_name} {mode}')
    report = evaluate_ir_from_files(output_file, test_data, k_list=(5, 10))

    metrics = {
        "Tokens_mean": float(np.mean(report["averages_tokens"])),            
        "NDCG@5":      report["summary"].get("avg_ndcg@5", 0.0),
        "NDCG@10":     report["summary"].get("avg_ndcg@10", 0.0),
        "PairwiseAcc": report["summary"].get("avg_pairacc", 0.0),
        "Top1":        report["summary"].get("avg_top1", 0.0),
        "SpearmanF":   report["summary"].get("avg_spearmanF", 0.0),
    }

    print(json.dumps(metrics, indent=2, ensure_ascii=False))
    detail_df = pd.DataFrame(report["detail"])
    detail_csv = os.path.join(saved_result_path, f"{system}_{mode}_ndcg.csv")
    detail_df.to_csv(detail_csv, index=False)
    np.save(os.path.join(saved_result_path, f'{system}_{mode}_averages.npy'), report["averages_tokens"])
    print(f"[Saved] per-sample metrics -> {detail_csv}")
    return detail_df, report

def get_label(cot_results: pd.DataFrame, direct_results: pd.DataFrame, k: int = 10):
  
    col = f'avg_ndcg@{k}' if f'avg_ndcg@{k}' in cot_results.columns else 'avg_ndcg'
    cot_avg_ndcg = cot_results[col].tolist()
    direct_avg_ndcg = direct_results[col].tolist()

    label = []
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            label.append(1)
        elif cot_avg_ndcg[i] < direct_avg_ndcg[i]:
            label.append(0)
        else:
            label.append(2)
    return label


In [ ]:
dataset_name = 'rank_ir'
results_csv = []
for model in ['Qwen3-4B']:
    mode = 'test'
    for system in ['ranking_direct_v1', 'ranking_CoT_v1']:
        PATH = os.path.join('/data/', f'{dataset_name}/{model}')
        data_path = os.path.join('/data/', f'{dataset_name}/processed_data')
        saved_file_path = os.path.join(PATH, 'saved_data')
        saved_result_path = os.path.join(PATH, 'saved_result')
        output_file = os.path.join(saved_file_path, f'{system}_{dataset_name}_{mode}_llm_output.jsonl')
        report = evaluate_ir_from_files(output_file, loaded_test_data, k_list=(5, 10))
        results_csv.append([report["summary"].get("avg_top1", 0.0),  report["summary"].get("avg_pairacc", 0.0), report["summary"].get("avg_ndcg@5", 0.0), report["summary"].get("avg_ndcg@10", 0.0), report["summary"].get("avg_spearmanF", 0.0), report["averages_tokens"]])


In [ ]:
results_csv = pd.DataFrame(results_csv, columns=['top1', 'pairacc', 'ndcg@5', 'ndcg@10', 'spearmanF', 'token'])
results_csv['token'] = results_csv['token'].apply(lambda x: np.mean(x))
results_csv

In [ ]:
def get_mixed_results(cot_results, direct_results, cot_tokens, direct_tokens):
    cot_avg_ndcg = cot_results['avg_ndcg@10'].tolist()
    direct_avg_ndcg = direct_results['avg_ndcg@10'].tolist()
    mixed_avg_ndcg = []
    mixed_tokens = []
    count_i = []
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            mixed_avg_ndcg.append(cot_avg_ndcg[i])
            mixed_tokens.append(cot_tokens[i])
            count_i.append(i)
        else:
            mixed_avg_ndcg.append(direct_avg_ndcg[i])
            mixed_tokens.append(direct_tokens[i])
    return mixed_avg_ndcg, mixed_tokens, count_i

mixed_avg_ndcg, mixed_tokens, count_i = get_mixed_results(detail_df_cot, detail_df_direct, report_cot["averages_tokens"], report_direct["averages_tokens"])
print(f'Mixed mean NDCG: {np.mean(mixed_avg_ndcg)}')
print(f'Mixed average tokens: {np.mean(mixed_tokens)}')

In [16]:
def compare_results(cot_results, direct_results):
    cot_avg_ndcg = cot_results['avg_ndcg@10'].tolist()
    direct_avg_ndcg = direct_results['avg_ndcg@10'].tolist()
    cot_count = 0
    direct_count = 0
    for i in range(len(cot_avg_ndcg)):
        if cot_avg_ndcg[i] > direct_avg_ndcg[i]:
            cot_count += 1
        elif cot_avg_ndcg[i] < direct_avg_ndcg[i]:
            direct_count += 1
    print(f'CoT count: {cot_count / len(cot_avg_ndcg)}, Direct count: {direct_count / len(direct_avg_ndcg)}')

compare_results(detail_df_cot, detail_df_direct)

CoT count: 0.4713766754353, Direct count: 0.4684955530502317
